In [ ]:
import scanpy as sc
import decoupler as dc
from rpy2.robjects.conversion import localconverter
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text
from sklearn.preprocessing import StandardScaler
import pandas as pd
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import seaborn as sns

%load_ext rpy2.ipython

In [ ]:
adata = sc.read("../results/adata/09-annotation.h5ad")

## Wilcoxon Sign Rank
### Cell Types

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden_0.5", method="wilcoxon")
sc.pl.rank_genes_groups(adata)

In [ ]:
sc.tl.filter_rank_genes_groups(adata, min_in_group_fraction=0.2,
                               max_out_group_fraction=0.2)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, groupby="leiden_0.5", n_genes=5, key="rank_genes_groups_filtered")

In [ ]:
df = sc.get.rank_genes_groups_df(adata, group=None)
df.to_csv("../results/diff_exp_cell_types.csv")

In [ ]:
for group in adata.uns["rank_genes_groups"]["names"].dtype.names:
    print(f"Cluster: {group}")
    sc.pl.umap(adata, ncols=5, color=["leiden_0.15"] + adata.uns["rank_genes_groups"]["names"][group][:4].tolist())


### Treatments

In [ ]:
macrophages = adata[adata.obs["cell_type"] == "macrophages"].copy()
macrophages.X = macrophages.layers["soupx_rounded"].copy()
macrophages.obs["day"] = macrophages.obs["day"].cat.rename_categories(str)
sc.pp.filter_genes(macrophages, min_counts=10)
sc.pp.normalize_total(macrophages)
sc.pp.log1p(macrophages)

In [ ]:
sc.tl.rank_genes_groups(macrophages, "day", method="wilcoxon")
sc.pl.rank_genes_groups(macrophages)

In [ ]:
sc.tl.filter_rank_genes_groups(macrophages, min_in_group_fraction=0.2,
                               max_out_group_fraction=0.2)

In [ ]:
sc.pl.rank_genes_groups_dotplot(macrophages, n_genes=5, dendrogram=False)

In [ ]:
df = sc.get.rank_genes_groups_df(macrophages, group=None)
df.to_csv("../results/diff_exp_macrophages_days.csv")

## Pseudobulk

In [ ]:
pdata = dc.pp.pseudobulk(
    macrophages,
    sample_col="sample",
    groups_col="day",
    layer="soupx_rounded",          
    mode="sum",
    skip_checks=True
)

In [ ]:
dc.pl.filter_samples(
    adata=pdata,
    groupby=["day", "sample"], 
    min_cells=5,
    min_counts=2000,
    figsize=(5, 8)
)

In [ ]:
dc.pp.filter_samples(
    pdata,
    min_cells=5,
    min_counts=2000)

In [ ]:
with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv["counts"] = pdata.to_df().astype(int).T
    ro.globalenv["meta"] = pdata.obs[["sample", "day"]].copy()
    ro.globalenv["outdir"] = "output"

In [ ]:
%%R
suppressMessages(library(DESeq2))

dds <- DESeqDataSetFromMatrix(
    countData=as.matrix(counts), 
    colData=meta, 
    design= ~ day)

dds <- DESeq(dds)

vst <- assay(vst(dds, blind = FALSE))
vst_df <- as.data.frame(vst)

# Dispersion Estimates
png("dispersions.png")
plotDispEsts(dds)
dev.off()


In [ ]:
%%R

print(resultsNames(dds))
res <- results(dds, name="day_2_vs_0")
res_df <- as.data.frame(res)
res_df$gene <- rownames(res_df)

In [ ]:
with localconverter(ro.default_converter + pandas2ri.converter):
    vst_df = ro.globalenv["vst_df"]
    res_df = ro.globalenv["res_df"]

In [ ]:
res_df.head()

In [ ]:
res_df = res_df.copy().dropna(subset=["log2FoldChange", "padj"])
    
# Clip padj to avoid log10(0) = -inf
res_df["-log10_padj"] = -np.log10(res_df["padj"].clip(lower=1e-300))
   
plt.figure()
plt.scatter(x=res_df["log2FoldChange"], y=res_df["-log10_padj"], s=1)
plt.xlabel("$log_{2}$ Fold Change")
plt.ylabel("-logFDR")
plt.axhline(-np.log10(0.05), color="grey", linestyle="--")

top = res_df.nsmallest(10, "padj")
texts = []
for _, row in top.iterrows():
    texts.append(plt.text(row["log2FoldChange"], row["-log10_padj"], row.name, fontsize=7))
adjust_text(texts, arrowprops=dict(arrowstyle="-", color="grey", lw=0.5))

plt.show()

In [ ]:
vst_top = vst_df.loc[res_df.index[:500].tolist()]
scaler = StandardScaler()
vst_scaled = pd.DataFrame(
    scaler.fit_transform(vst_top.T).T,
    index=vst_top.index,
    columns=vst_top.columns)

dist = pdist(vst_scaled, metric="correlation")
Z = linkage(dist, method="average")
g = sns.clustermap(
    vst_scaled,
    row_linkage=Z,              
    col_cluster=True,
    metric='correlation',
    method='average',
    # cmap='RdYlBu_r',
    figsize=(14, 10),
    yticklabels=False)
plt.show()